# Transformer 机器翻译练习版

这个 notebook 用于课堂练习，基于 `MT-transformer-5.ipynb` 改写而来。

本版本只保留 10 个核心 TODO，重点练习机器翻译流程中最重要的部分：

1. 数据路径与读取
2. 分词
3. 词表和 token id
4. padding 与 DataLoader
5. Transformer 关键模块
6. 训练输入/目标错位
7. 推理生成

其余辅助代码已经给出，学生补全 TODO 后即可运行完整流程。


In [ ]:
# pip install torch numpy
# 如果使用 Ascend NPU，需要环境中已安装对应版本的 torch_npu


In [1]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

# 设置随机种子以确保可重复性
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

# 指定运行设备：'auto'、'npu'、'cuda'、'mps'、'cpu'
# 在 Ascend 环境中使用 NPU 时，可以改成 DEVICE_NAME = 'npu'
DEVICE_NAME = 'auto'
NPU_ID = 0

def get_device(device_name='auto', npu_id=0):
    if device_name in ('auto', 'npu'):
        try:
            import torch_npu  # noqa: F401
            if hasattr(torch, 'npu') and torch.npu.is_available():
                device = torch.device(f'npu:{npu_id}')
                torch.npu.set_device(f'npu:{npu_id}')
                return device
        except ImportError:
            if device_name == 'npu':
                raise RuntimeError('已指定 DEVICE_NAME="npu"，但当前环境没有安装 torch_npu')
        if device_name == 'npu':
            raise RuntimeError('已指定 DEVICE_NAME="npu"，但 torch.npu.is_available() 不是 True')

    if device_name in ('auto', 'cuda') and torch.cuda.is_available():
        return torch.device('cuda')
    if device_name == 'cuda':
        raise RuntimeError('已指定 DEVICE_NAME="cuda"，但 CUDA 不可用')

    if device_name in ('auto', 'mps') and hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    if device_name == 'mps':
        raise RuntimeError('已指定 DEVICE_NAME="mps"，但 MPS 不可用')

    return torch.device('cpu')

device = get_device(DEVICE_NAME, NPU_ID)
if device.type == 'npu':
    torch.npu.manual_seed_all(42)
elif device.type == 'cuda':
    torch.cuda.manual_seed_all(42)
print(f'Using device: {device}')

# TODO 1: 指定数据目录和中英文文件路径
# 提示：DATA_DIR = Path('data')，中文文件是 chinese.txt，英文文件是 english.txt
DATA_DIR = Path('data')
CHINESE_PATH = DATA_DIR / 'chinese.txt'
ENGLISH_PATH = DATA_DIR / 'english.txt'

MAX_SAMPLES = 2000

def load_parallel_corpus(chinese_path, english_path, max_samples=None):
    chinese_lines = chinese_path.read_text(encoding='utf-8').splitlines()
    english_lines = english_path.read_text(encoding='utf-8').splitlines()

    if len(chinese_lines) != len(english_lines):
        raise ValueError(f'中英文行数不一致: {len(chinese_lines)} vs {len(english_lines)}')

    pairs = [
        (chinese.strip(), english.strip())
        for chinese, english in zip(chinese_lines, english_lines)
        if chinese.strip() and english.strip()
    ]
    if max_samples is not None:
        pairs = pairs[:max_samples]

    chinese_sentences = [chinese for chinese, _ in pairs]
    english_sentences = [english for _, english in pairs]
    return chinese_sentences, english_sentences

chinese_sentences, english_sentences = load_parallel_corpus(CHINESE_PATH, ENGLISH_PATH, MAX_SAMPLES)
print(f'Loaded {len(chinese_sentences)} sentence pairs')
print(chinese_sentences[0])
print(english_sentences[0])


Using device: cuda
Loaded 2000 sentence pairs
1998年 , 经过 统一 部署 , 伊犁州 , 地 两 级 党委 开始 尝试 以 宣讲 团 的 形式 , 深入 学校 , 村民 院落 , 田间 地头 , 向 各族 群众 进行 面对面 宣讲 .
in 1998 , the yili autonomous prefecture cpc committee and the yili prefecture cpc committee made unified arrangements and sent on a trial basis several propaganda teams deep into the schools , villagers ' courtyards , and fields to carry out face - to - face propaganda among the people of all nationalities .


In [2]:
# data 目录中的语料已经用空格完成分词，这里直接按空格切分。
def tokenize_ch(text):
    # TODO 2: 按空格切分中英文句子，返回 token 列表
    return text.split()

def tokenize_en(text):
    return text.split()

from collections import Counter

def build_vocab(data, min_freq=1):
    counter = Counter()
    for tokens in data:
        counter.update(tokens)

    # TODO 3: 为普通 token 分配 id，id 从 4 开始
    vocab = {}
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab) + 4

    vocab['<pad>'] = 0
    vocab['<sos>'] = 1
    vocab['<eos>'] = 2
    vocab['<unk>'] = 3
    return vocab

chinese_vocab = build_vocab([tokenize_ch(s) for s in chinese_sentences])
english_vocab = build_vocab([tokenize_en(s) for s in english_sentences])

def sentence_to_indices(sentence, vocab):
    # TODO 4: 在句首加入 <sos>，句尾加入 <eos>，未知词使用 <unk>
    sentence = ['<sos>'] + sentence + ['<eos>']
    indices = [vocab.get(token, vocab['<unk>']) for token in sentence]
    return indices

data = [
    (
        sentence_to_indices(tokenize_ch(chinese), chinese_vocab),
        sentence_to_indices(tokenize_en(english), english_vocab),
    )
    for chinese, english in zip(chinese_sentences, english_sentences)
]

print(f'Chinese vocab size: {len(chinese_vocab)}')
print(f'English vocab size: {len(english_vocab)}')
print('First indexed pair:', data[0])


Chinese vocab size: 7795
English vocab size: 6466
First indexed pair: ([1, 4, 5, 6, 7, 8, 5, 9, 5, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 5, 21, 22, 5, 23, 24, 5, 25, 26, 5, 27, 28, 29, 30, 31, 17, 32, 2], [1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 7, 8, 10, 11, 12, 14, 15, 16, 13, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 7, 27, 6, 28, 29, 30, 6, 13, 31, 32, 33, 34, 35, 36, 32, 36, 35, 23, 37, 7, 38, 39, 40, 41, 42, 2])


In [3]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)

    # TODO 5: 使用 pad_sequence 补齐中文和英文 batch，padding_value=0, batch_first=False
    src_pad = pad_sequence(
        [torch.tensor(seq) for seq in src_batch], 
        padding_value=0, 
        batch_first=False,
    )

    trg_pad = pad_sequence(
        [torch.tensor(seq) for seq in trg_batch], 
        padding_value=0, 
        batch_first=False,
    )

    return src_pad, trg_pad

class TranslationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

dataset = TranslationDataset(data)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)


In [4]:
import math

class Transformer(nn.Module):
    def __init__(self, input_dim, output_dim, d_model, nhead, num_encoder_layers, num_decoder_layers, dim_feedforward, dropout):
        super().__init__()

        # TODO 6: 定义源语言 embedding、目标语言 embedding 和输出线性层
        self.src_embedding = nn.Embedding(input_dim, d_model)
        self.trg_embedding = nn.Embedding(output_dim, d_model)

        self.d_model = d_model
        self.transformer = nn.Transformer(
            d_model,
            nhead,
            num_encoder_layers,
            num_decoder_layers,
            dim_feedforward,
            dropout,
            batch_first=False,
        )

        self.fc_out = nn.Linear(d_model, output_dim)
        self.dropout = nn.Dropout(dropout)

    def _generate_positional_encoding(self, seq_len):
        position = torch.arange(seq_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, self.d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / self.d_model)
        )
        pe = torch.zeros(seq_len, self.d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(1)

    def forward(self, src, trg, trg_mask=None, padding_mask=None):
        src_seq_length, N = src.shape
        trg_seq_length, N = trg.shape

        src_pos = self._generate_positional_encoding(src_seq_length).to(src.device)
        trg_pos = self._generate_positional_encoding(trg_seq_length).to(trg.device)

        src_pos = src_pos.expand(-1, N, -1)
        trg_pos = trg_pos.expand(-1, N, -1)

        # TODO 7: 对 src 和 trg 做 embedding，加位置编码，再 dropout
        src = self.src_embedding(src) * math.sqrt(self.d_model) + src_pos
        trg = self.trg_embedding(trg) * math.sqrt(self.d_model) + trg_pos
        src = self.dropout(src)
        trg = self.dropout(trg)

        output = self.transformer(
            src,
            trg,
            tgt_mask=trg_mask,
            tgt_key_padding_mask=padding_mask,
        )
        prediction = self.fc_out(output)
        return prediction


In [7]:
INPUT_DIM = len(chinese_vocab)
OUTPUT_DIM = len(english_vocab)
D_MODEL = 32
NHEAD = 2
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
DIM_FEEDFORWARD = 32
DROPOUT = 0.05
EPOCHS = 100

model = Transformer(
    INPUT_DIM,
    OUTPUT_DIM,
    D_MODEL,
    NHEAD,
    NUM_ENCODER_LAYERS,
    NUM_DECODER_LAYERS,
    DIM_FEEDFORWARD,
    DROPOUT,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(EPOCHS):
    for i, (src, trg) in enumerate(dataloader):
        src = src.to(device)
        trg = trg.to(device)

        # TODO 8: 构造 trg_input 和 trg_expected
        # 提示：trg_input 去掉最后一个 token，trg_expected 去掉第一个 token
        trg_input = trg[:-1, :]
        trg_expected = trg[1:, :]

        trg_mask = nn.Transformer.generate_square_subsequent_mask(trg_input.size(0)).to(device).bool()
        padding_mask = (trg_input == 0).transpose(0, 1)

        # TODO 9: 前向传播并计算交叉熵损失
        output = model(src, trg_input, trg_mask=trg_mask, padding_mask=padding_mask)

        # 提示：output reshape 成 [-1, OUTPUT_DIM]，trg_expected reshape 成 [-1]
        loss = criterion(output.reshape(-1, OUTPUT_DIM), trg_expected.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch {epoch + 1}, Loss: {loss.item()}')


c:\Users\zyh07\Code\Project\AI_Programming\.venv\Lib\site-packages\torch\nn\modules\transformer.py:143: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


Epoch 1, Loss: 6.600331783294678
Epoch 2, Loss: 5.990771770477295
Epoch 3, Loss: 6.022068977355957
Epoch 4, Loss: 5.933605670928955
Epoch 5, Loss: 5.9345927238464355
Epoch 6, Loss: 5.685370445251465
Epoch 7, Loss: 5.136136531829834
Epoch 8, Loss: 5.3049445152282715
Epoch 9, Loss: 5.217198371887207
Epoch 10, Loss: 5.229440212249756
Epoch 11, Loss: 5.041036605834961
Epoch 12, Loss: 5.047039985656738
Epoch 13, Loss: 4.978816986083984
Epoch 14, Loss: 4.759888172149658
Epoch 15, Loss: 4.717620849609375
Epoch 16, Loss: 4.934726238250732
Epoch 17, Loss: 4.238602638244629
Epoch 18, Loss: 4.30856990814209
Epoch 19, Loss: 4.086709976196289
Epoch 20, Loss: 4.504942417144775
Epoch 21, Loss: 3.9749693870544434
Epoch 22, Loss: 4.1931023597717285
Epoch 23, Loss: 4.202747344970703
Epoch 24, Loss: 4.022885799407959
Epoch 25, Loss: 3.850825786590576
Epoch 26, Loss: 3.8817050457000732
Epoch 27, Loss: 3.698814868927002
Epoch 28, Loss: 3.7765326499938965
Epoch 29, Loss: 3.6766366958618164
Epoch 30, Loss: 3

In [8]:
def translate_sentence(sentence, src_vocab, trg_vocab, model, device=None, max_len=50):
    model.eval()
    if device is None:
        device = next(model.parameters()).device
    tokens = tokenize_ch(sentence)
    indices = sentence_to_indices(tokens, src_vocab)    
    src_tensor = torch.tensor(indices, dtype=torch.long, device=device).unsqueeze(1)
    trg_indices = [trg_vocab['<sos>']]
    
    for i in range(max_len):
        trg_tensor = torch.tensor(trg_indices, dtype=torch.long, device=device).unsqueeze(1)
        with torch.no_grad():
            output = model(src_tensor, trg_tensor)
            #print("output:",output.argmax(2))
        pred_token = output.argmax(2)[-1].item()
        #print("pred:",pred_token)
        trg_indices.append(pred_token)
        if pred_token == trg_vocab['<eos>']:
            break
    
    idx_to_trg = {idx: token for token, idx in trg_vocab.items()}
    trg_tokens = [idx_to_trg.get(i, '<unk>') for i in trg_indices]
    if trg_tokens and trg_tokens[0] == '<sos>':
        trg_tokens = trg_tokens[1:]
    if '<eos>' in trg_tokens:
        trg_tokens = trg_tokens[:trg_tokens.index('<eos>')]
    return ' '.join(trg_tokens)

# BLEU 评测：纯 Python 实现，避免额外依赖 nltk / sacrebleu。
from collections import Counter
import math


def get_ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def compute_bleu(references, hypotheses, max_n=4, smooth=True):
    """
    计算 corpus-level BLEU。

    Args:
        references: List[List[str]]，每个元素是参考译文 token 列表。
        hypotheses: List[List[str]]，每个元素是模型译文 token 列表。
        max_n: BLEU-n 的最大 n-gram 阶数，默认 BLEU-4。
        smooth: 是否做平滑，避免高阶 n-gram 命中为 0 时 BLEU 直接变成 0。
    """
    clipped_counts = [0] * max_n
    total_counts = [0] * max_n
    ref_len = 0
    hyp_len = 0

    for ref, hyp in zip(references, hypotheses):
        ref_len += len(ref)
        hyp_len += len(hyp)
        for n in range(1, max_n + 1):
            ref_ngrams = get_ngrams(ref, n)
            hyp_ngrams = get_ngrams(hyp, n)
            clipped_counts[n - 1] += sum(
                min(count, ref_ngrams.get(ngram, 0))
                for ngram, count in hyp_ngrams.items()
            )
            total_counts[n - 1] += max(len(hyp) - n + 1, 0)

    if hyp_len == 0:
        return 0.0

    precisions = []
    for match_count, total_count in zip(clipped_counts, total_counts):
        if smooth:
            precisions.append((match_count + 1) / (total_count + 1))
        else:
            precisions.append(match_count / total_count if total_count > 0 else 0.0)

    if min(precisions) <= 0:
        return 0.0

    brevity_penalty = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / hyp_len)
    geometric_mean = math.exp(sum(math.log(p) for p in precisions) / max_n)
    return brevity_penalty * geometric_mean


def evaluate_bleu(source_chinese_sentences, reference_english_sentences, model, src_vocab, trg_vocab, device=None, max_len=50, eval_size=None):
    """
    在指定数据上评测 BLEU。

    reference_english_sentences: GT 英文句子，作为 BLEU reference。
    predicted_english_sentences: 模型根据中文输入生成的英文句子，作为 BLEU hypothesis。
    eval_size=None 表示使用全部样本；为了快速调试，可以设置 eval_size=100。
    """
    if eval_size is None:
        eval_chinese = source_chinese_sentences
        eval_reference_english = reference_english_sentences
    else:
        eval_chinese = source_chinese_sentences[:eval_size]
        eval_reference_english = reference_english_sentences[:eval_size]

    predicted_english_sentences = []
    references = []
    hypotheses = []

    for src_sentence, gt_english_sentence in zip(eval_chinese, eval_reference_english):
        pred_english_sentence = translate_sentence(
            src_sentence,
            src_vocab,
            trg_vocab,
            model,
            device=device,
            max_len=max_len,
        )
        predicted_english_sentences.append(pred_english_sentence)
        references.append(tokenize_en(gt_english_sentence))
        hypotheses.append(tokenize_en(pred_english_sentence))

    bleu = compute_bleu(references, hypotheses, max_n=4, smooth=True)
    return bleu, eval_reference_english, predicted_english_sentences, references, hypotheses


# 评测 BLEU-4
EVAL_SIZE = 100
bleu_score, reference_english_sentences, predicted_english_sentences, references, hypotheses = evaluate_bleu(
    chinese_sentences,
    english_sentences,  # GT 英文，只作为 reference
    model,
    chinese_vocab,
    english_vocab,
    device=device,
    max_len=50,
    eval_size=EVAL_SIZE,
)

print(f'BLEU-4 score on {EVAL_SIZE if EVAL_SIZE is not None else len(chinese_sentences)} samples: {bleu_score:.4f}')
print(f'BLEU-4 percentage: {bleu_score * 100:.2f}')
print('\n前 5 条评测样例：')
for i in range(min(5, len(predicted_english_sentences))):
    print(f'[{i}] 中文输入: {chinese_sentences[i]}')
    print(f'    GT 英文: {reference_english_sentences[i]}')
    print(f'    模型翻译: {predicted_english_sentences[i]}')



BLEU-4 score on 100 samples: 0.1056
BLEU-4 percentage: 10.56

前 5 条评测样例：
[0] 中文输入: 1998年 , 经过 统一 部署 , 伊犁州 , 地 两 级 党委 开始 尝试 以 宣讲 团 的 形式 , 深入 学校 , 村民 院落 , 田间 地头 , 向 各族 群众 进行 面对面 宣讲 .
    GT 英文: in 1998 , the yili autonomous prefecture cpc committee and the yili prefecture cpc committee made unified arrangements and sent on a trial basis several propaganda teams deep into the schools , villagers ' courtyards , and fields to carry out face - to - face propaganda among the people of all nationalities .
    模型翻译: in the yili prefecture cpc central committee , villagers ' courtyards , and the yili hardships to carry out the yili prefecture investigation , and the yili hardships to carry out the yili prefecture investigation of the propaganda teams deep , and the yili prefecture investigation of the propaganda teams
[1] 中文输入: 在 不少 乡村 , 群众 在 草地 , 球场 , 树 荫 下 席地而坐 , 认真 听讲 , 参与者 少 则 数 十 人 , 多 则 千 余 人 .
    GT 英文: in many rural areas , the masses sat on the grassland , in the football fields , or u